# GUV Aggregation — Imaging Flow Cytometry Analysis
**Cy5 channel | Area AND Width classification | NC-anchored thresholds**

---
### How to use
1. Edit **Cell 2** only — set your file paths and condition label
2. Run All Cells

### Sections
| Section | Content |
|---------|---------|
| **1** | Parameter overview — all available parameters |
| **2** | Classification scatter: Area vs Width |
| **3** | Cross-dataset comparison + paired t-test |
| **4** | Summary table + GraphPad CSV |

### Classification logic
An event is classified as **aggregate-like** if:
- **Area** > `AREA_MULTIPLIER` × median area of round NC events (AR > 0.85)
- **AND Width** > NC `WID_PCT`th percentile

Both thresholds are calibrated independently per dataset from the negative control.

### Requirements
```
pip install pandas numpy matplotlib scipy
```
---


In [ ]:
# ============================================================
#  CONFIGURATION — only edit this cell
# ============================================================

# One entry per independent experiment:
# ("Label", "path/to/negative_control.txt", "path/to/treated.txt")
DATASETS = [
    ("Experiment 1", "/path/to/neg_control_1.txt", "/path/to/treated_1.txt"),
    ("Experiment 2", "/path/to/neg_control_2.txt", "/path/to/treated_2.txt"),
    ("Experiment 3", "/path/to/neg_control_3.txt", "/path/to/treated_3.txt"),
]

CONDITION_LABEL  = "5 µM IgLON1"  # label for treated condition
AREA_MULTIPLIER  = 2.0             # area threshold = AREA_MULTIPLIER × median single GUV
WID_PCT          = 85              # width threshold = NC Pth percentile

# ============================================================
print(f"✓  {len(DATASETS)} datasets configured")
for name, nc_p, tr_p in DATASETS:
    print(f"   {name}")
    print(f"     NC: {nc_p.split('/')[-1]}")
    print(f"     TR: {tr_p.split('/')[-1]}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

# ── Colour palette — consistent across all plots ─────────────────────────
C_SING    = "#5fa85a"   # green  — single-like events (both panels)
C_AGG     = "#7c5cbf"   # purple — aggregate-like events (both panels)
C_THRESH  = "#e05555"   # red    — threshold lines
C_NC_HIST = "#8fcfaa"   # light green  — NC histograms
C_TR_HIST = "#c3aef0"   # light purple — treated histograms
C_NC_BAR  = "#3a8a5c"   # dark green  — NC bars
C_TR_BAR  = "#7c5cbf"   # dark purple — treated bars
C_GOOD    = "#2e7d32"
C_BAD     = "#c62828"

# ── Helpers ───────────────────────────────────────────────────────────────
def load(path):
    df = pd.read_csv(path, sep="\t", skiprows=1, low_memory=False)
    df.dropna(how="all", inplace=True)
    for col in df.columns:
        if col != "Object Number":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def detect_columns(df):
    """Auto-detect mask ID from column names. Works for M04, M05, etc."""
    cols = df.columns.tolist()
    area_cols = [c for c in cols if c.startswith("Area_M") and "_Ch" not in c]
    if not area_cols:
        raise ValueError(f"No Area column found. Available: {cols[:10]}")
    mask_id = area_cols[0].replace("Area_", "")
    int_cols = [c for c in cols if "Intensity_MC_Ch" in c
                and "Bright" not in c and "Aspect" not in c]
    ch_id = int_cols[0].split("_Ch")[-1] if int_cols else None
    return {
        "AREA": f"Area_{mask_id}",
        "AR":   f"Aspect Ratio_{mask_id}",
        "WID":  f"Width_{mask_id}",
        "LEN":  f"Length_{mask_id}",
        "INT":  f"Intensity_MC_Ch{ch_id}" if ch_id else None,
        "BDI":  next((c for c in cols if "Bright Detail Intensity R3" in c), None),
        "ARI":  next((c for c in cols if "Aspect Ratio Intensity" in c), None),
        "CONT": next((c for c in cols if "Contrast_" in c), None),
    }, mask_id

def get_thresholds(nc, col_map):
    rnd = nc[nc[col_map["AR"]] > 0.85][col_map["AREA"]].dropna() \
          if col_map["AR"] in nc.columns else nc[col_map["AREA"]].dropna()
    med    = rnd.median() if len(rnd) >= 50 else nc[col_map["AREA"]].median()
    area_t = AREA_MULTIPLIER * med
    wid_t  = np.percentile(nc[col_map["WID"]].dropna(), WID_PCT)
    return area_t, wid_t, med

def classify(df, col_map, area_t, wid_t):
    return (df[col_map["AREA"]] > area_t) & (df[col_map["WID"]] > wid_t)

def ds(d, n=3000):
    return d.sample(min(n, len(d)), random_state=42) if len(d) > n else d

print("✓  Libraries and helpers loaded")


In [ ]:
results = []

for name, nc_path, tr_path in DATASETS:
    print(f"Loading {name}...")
    try:
        nc = load(nc_path); tr = load(tr_path)
    except FileNotFoundError as e:
        print(f"  ✗  {e}"); continue

    col_map, mask_id       = detect_columns(nc)
    area_t, wid_t, med_s   = get_thresholds(nc, col_map)
    nc_agg                 = classify(nc, col_map, area_t, wid_t)
    tr_agg                 = classify(tr, col_map, area_t, wid_t)
    ai_nc                  = nc_agg.mean() * 100
    ai_tr                  = tr_agg.mean() * 100
    delta                  = ai_tr - ai_nc
    fold                   = ai_tr / ai_nc if ai_nc > 0 else float("inf")

    print(f"  Channel: {mask_id}")
    print(f"  Median single GUV area: {med_s:.1f} µm²  "
          f"(d ≈ {2*np.sqrt(med_s/np.pi):.1f} µm)")
    print(f"  Area threshold ({AREA_MULTIPLIER}×): {area_t:.1f} µm²")
    print(f"  Width threshold (P{WID_PCT}): {wid_t:.2f} µm")
    print(f"  NC={ai_nc:.2f}%  TR={ai_tr:.2f}%  "
          f"Δ={delta:+.2f}%  fold={fold:.2f}×")

    results.append(dict(name=name, nc=nc, tr=tr,
                        nc_agg=nc_agg, tr_agg=tr_agg,
                        ai_nc=ai_nc, ai_tr=ai_tr,
                        delta=delta, fold=fold,
                        area_t=area_t, wid_t=wid_t,
                        med_s=med_s, col_map=col_map, mask_id=mask_id))

if len(results) >= 2:
    nc_ais = [r["ai_nc"] for r in results]
    tr_ais = [r["ai_tr"] for r in results]
    t_stat, p_val = ttest_rel(tr_ais, nc_ais)
    diffs = [t-n for t,n in zip(tr_ais, nc_ais)]
    sig = "***" if p_val<0.001 else ("**" if p_val<0.01 else ("*" if p_val<0.05 else "ns"))
    print(f"\n── Paired t-test (n={len(results)}) ──────────────────────")
    print(f"  Diffs: {[round(d,2) for d in diffs]}")
    print(f"  Mean Δ = {np.mean(diffs):.2f}%   SD = {np.std(diffs,ddof=1):.2f}%")
    print(f"  t = {t_stat:.4f}   p = {p_val:.4f}   {sig}")

print(f"\n✓  {len(results)} datasets loaded")


---
## Section 1 — Parameter Overview
All available parameters. **Δ%** = relative mean difference (NC vs treated). ★ = used for classification.


In [ ]:
for r in results:
    cm = r["col_map"]
    overview = [
        (cm["AREA"],"Area (µm²) ★"), (cm["WID"],"Width (µm) ★"),
        (cm["INT"], "Intensity MC"),  (cm["BDI"],"BDI R3"),
        (cm["LEN"], "Length (µm)"),   (cm["AR"], "Aspect Ratio"),
        (cm["ARI"], "AR Intensity"),  (cm["CONT"],"Contrast"),
    ]
    params = [(c,l) for c,l in overview
              if c and c in r["nc"].columns and c in r["tr"].columns]
    ncols = 4; nrows = int(np.ceil(len(params)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.2*nrows))
    fig.patch.set_facecolor("white"); axes = axes.flatten()

    for i, (col, label) in enumerate(params):
        ax = axes[i]
        ax.set_facecolor("white"); ax.spines[["top","right"]].set_visible(False)
        ax.spines[["left","bottom"]].set_color("#bbbbbb")
        all_v = pd.concat([r["nc"][col].dropna(), r["tr"][col].dropna()])
        xmax  = all_v.quantile(0.98)
        bins  = np.linspace(max(all_v.min(), 0), xmax, 55)
        ax.hist(r["nc"][col].clip(upper=xmax), bins=bins, color=C_NC_HIST,
                alpha=0.75, label="NC", edgecolor="none", density=True)
        ax.hist(r["tr"][col].clip(upper=xmax), bins=bins, color=C_TR_HIST,
                alpha=0.65, label=CONDITION_LABEL, edgecolor="none", density=True)
        if col == cm["AREA"]:
            ax.axvline(r["area_t"], color=C_THRESH, lw=1.6, ls="--",
                       label=f"={r['area_t']:.1f} µm²")
        elif col == cm["WID"]:
            ax.axvline(r["wid_t"], color=C_THRESH, lw=1.6, ls="--",
                       label=f"={r['wid_t']:.1f} µm")
        nc_m = r["nc"][col].dropna().mean()
        tr_m = r["tr"][col].dropna().mean()
        dp   = 100*(tr_m-nc_m)/nc_m if nc_m != 0 else 0
        dc   = C_GOOD if dp>5 else (C_BAD if dp<-5 else "#888888")
        ax.text(0.97,0.97,f"Δ={dp:+.1f}%",transform=ax.transAxes,
                ha="right",va="top",fontsize=10,fontweight="bold",color=dc)
        ax.set_title(label, fontsize=11, fontweight="bold",
                     color=C_THRESH if "★" in label else "#1B2A4A")
        ax.set_xlabel(label, fontsize=9, color="#333333")
        ax.set_ylabel("Density", fontsize=9, color="#333333")
        ax.legend(fontsize=7.5, framealpha=0.9, edgecolor="#dddddd")

    for j in range(i+1, len(axes)): axes[j].set_visible(False)
    fig.suptitle(f"{r['name']} — Parameter Overview  |  "
                 f"Green = NC, Purple = {CONDITION_LABEL}",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout(); plt.show()


---
## Section 2 — Classification Scatter: Area vs Width
**Same green and purple in both panels.** Red dashed lines = NC-derived thresholds (values in figure caption).


In [ ]:
def plot_scatter_clean(ax, df, is_agg, ai_val, label, col_map, area_t, wid_t):
    ax.set_facecolor("white")
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["left","bottom"]].set_color("#bbbbbb")
    ax.tick_params(colors="#444444", labelsize=10)

    sing = ds(df[~is_agg]); agg = ds(df[is_agg], 1500)
    xmax = df[col_map["AREA"]].quantile(0.995)*1.05
    ymax = df[col_map["WID"]].quantile(0.995)*1.2
    ymin = max(df[col_map["WID"]][df[col_map["WID"]]>0].min(), 0.1)

    # Same colours in both panels
    ax.scatter(sing[col_map["AREA"]], sing[col_map["WID"]],
               color=C_SING, alpha=0.35, s=6, rasterized=True, linewidths=0)
    ax.scatter(agg[col_map["AREA"]],  agg[col_map["WID"]],
               color=C_AGG,  alpha=0.75, s=8, rasterized=True, linewidths=0)

    ax.axvline(area_t, color=C_THRESH, lw=1.8, ls="--", zorder=5)
    ax.axhline(wid_t,  color=C_THRESH, lw=1.8, ls="--", zorder=5)
    ax.fill_betweenx([wid_t, ymax], area_t, xmax,
                     alpha=0.07, color=C_AGG, zorder=0)

    ax.text(0.97, 0.97, f"AI = {ai_val:.2f}%",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=12, fontweight="bold", color="#333333",
            bbox=dict(boxstyle="round,pad=0.3", fc="white",
                      ec="#aaaaaa", alpha=0.95))

    ax.set_xlim(0, xmax); ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Area of Cy5 membrane signal (µm²)",
                  fontsize=11, color="#333333")
    ax.set_ylabel("Width of Cy5 membrane signal (µm)",
                  fontsize=11, color="#333333")
    ax.set_title(label, fontsize=12, fontweight="bold",
                 color="#1B2A4A", pad=10)
    ax.legend(handles=[
        Patch(facecolor=C_SING, alpha=0.8,
              label=f"Single-like ({(~is_agg).sum():,})"),
        Patch(facecolor=C_AGG,  alpha=0.9,
              label=f"Aggregate-like ({is_agg.sum():,})"),
        Line2D([0],[0], color=C_THRESH, ls="--", lw=1.5,
               label="NC-derived thresholds"),
    ], fontsize=9, framealpha=0.95, edgecolor="#dddddd", loc="upper left")

for r in results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor("white")
    plot_scatter_clean(axes[0], r["nc"], r["nc_agg"], r["ai_nc"],
                       f"{r['name']} — Negative control",
                       r["col_map"], r["area_t"], r["wid_t"])
    plot_scatter_clean(axes[1], r["tr"], r["tr_agg"], r["ai_tr"],
                       f"{r['name']} — {CONDITION_LABEL}",
                       r["col_map"], r["area_t"], r["wid_t"])
    plt.tight_layout(pad=2.0); plt.show()


---
## Section 3 — Cross-Dataset Comparison


In [ ]:
if len(results) < 2:
    print("Need at least 2 datasets for comparison.")
else:
    names  = [r["name"]  for r in results]
    ai_ncs = [r["ai_nc"] for r in results]
    ai_trs = [r["ai_tr"] for r in results]
    deltas = [r["delta"] for r in results]
    folds  = [r["fold"]  for r in results]
    n      = len(results)

    t_stat, p_val = ttest_rel(ai_trs, ai_ncs)
    diffs = [t-nc for t,nc in zip(ai_trs,ai_ncs)]
    sig = "***" if p_val<0.001 else ("**" if p_val<0.01 else ("*" if p_val<0.05 else "ns"))

    fig = plt.figure(figsize=(16,10)); fig.patch.set_facecolor("white")
    gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.45,wspace=0.38)
    ax_nc=fig.add_subplot(gs[0,0]); ax_tr=fig.add_subplot(gs[0,1])
    ax_both=fig.add_subplot(gs[0,2]); ax_d=fig.add_subplot(gs[1,0])
    ax_f=fig.add_subplot(gs[1,1]);   ax_sum=fig.add_subplot(gs[1,2])
    for ax in [ax_nc,ax_tr,ax_both,ax_d,ax_f]:
        ax.set_facecolor("white"); ax.spines[["top","right"]].set_visible(False)
        ax.spines[["left","bottom"]].set_color("#bbbbbb")

    def lbars(ax,bars,vals):
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.1,
                    f"{val:.2f}%",ha="center",va="bottom",
                    fontsize=10,fontweight="bold",color="#333333")

    bars=ax_nc.bar(names,ai_ncs,color=C_NC_BAR,alpha=0.85,edgecolor="white",width=0.5)
    lbars(ax_nc,bars,ai_ncs); ax_nc.set_ylim(0,max(ai_ncs)*1.6+1)
    ax_nc.set_ylabel("Aggregation Index (%)",fontsize=11)
    ax_nc.set_title("Negative Control",fontsize=12,fontweight="bold")

    bars2=ax_tr.bar(names,ai_trs,color=C_TR_BAR,alpha=0.85,edgecolor="white",width=0.5)
    lbars(ax_tr,bars2,ai_trs); ax_tr.set_ylim(0,max(ai_trs)*1.6+1)
    ax_tr.set_ylabel("Aggregation Index (%)",fontsize=11)
    ax_tr.set_title(CONDITION_LABEL,fontsize=12,fontweight="bold")

    x=np.arange(n); w=0.35
    ax_both.bar(x-w/2,ai_ncs,w,color=C_NC_BAR,alpha=0.85,edgecolor="white",label="NC")
    ax_both.bar(x+w/2,ai_trs,w,color=C_TR_BAR,alpha=0.85,edgecolor="white",label=CONDITION_LABEL)
    ax_both.set_xticks(x); ax_both.set_xticklabels(names)
    ax_both.set_ylim(0,max(ai_ncs+ai_trs)*1.6+1)
    ax_both.set_ylabel("Aggregation Index (%)",fontsize=11)
    ax_both.set_title("NC vs Treated",fontsize=12,fontweight="bold")
    ax_both.legend(fontsize=9,framealpha=0.9,edgecolor="#dddddd")
    yb=max(ai_trs)*1.3
    for i,(nv,tv) in enumerate(zip(ai_ncs,ai_trs)):
        ax_both.plot([i-w/2,i-w/2,i+w/2,i+w/2],[nv+0.3,yb,yb,tv+0.3],color="#333333",lw=1)
        ax_both.text(i,yb+0.2,sig,ha="center",fontsize=12,fontweight="bold")

    d_cols=[C_GOOD if d>0 else C_BAD for d in deltas]
    bars3=ax_d.bar(names,deltas,color=d_cols,alpha=0.85,edgecolor="white",width=0.5)
    for bar,val in zip(bars3,deltas):
        ax_d.text(bar.get_x()+bar.get_width()/2,
                  bar.get_height()+0.05 if val>=0 else bar.get_height()-0.3,
                  f"{val:+.2f}%",ha="center",va="bottom",
                  fontsize=10,fontweight="bold",color="#333333")
    ax_d.axhline(0,color="#888888",lw=1,ls="--")
    ax_d.set_ylim(min(deltas)-2,max(deltas)*1.6+1)
    ax_d.set_ylabel("Δ AI  (treated − NC)  (%)",fontsize=11)
    ax_d.set_title("Delta Aggregation Index",fontsize=12,fontweight="bold")

    f_cols=[C_GOOD if f>1 else C_BAD for f in folds]
    bars4=ax_f.bar(names,folds,color=f_cols,alpha=0.85,edgecolor="white",width=0.5)
    for bar,val in zip(bars4,folds):
        ax_f.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
                  f"{val:.2f}×",ha="center",va="bottom",
                  fontsize=10,fontweight="bold",color="#333333")
    ax_f.axhline(1,color="#888888",lw=1,ls="--",label="No effect (1×)")
    ax_f.set_ylim(0,max(folds)*1.6+0.2)
    ax_f.set_ylabel("Fold increase (treated / NC)",fontsize=11)
    ax_f.set_title("Fold Increase",fontsize=12,fontweight="bold")
    ax_f.legend(fontsize=8,framealpha=0.9)

    ax_sum.axis("off"); ax_sum.set_title("Summary",fontsize=12,fontweight="bold")
    consistent=all(d>0 for d in deltas); s_col=C_GOOD if consistent else C_BAD
    ax_sum.text(0.5,0.90,
                "All datasets show\naggregation increase ✓" if consistent
                else "Not all datasets agree ⚠",
                transform=ax_sum.transAxes,ha="center",va="top",
                fontsize=13,fontweight="bold",color=s_col,
                bbox=dict(boxstyle="round,pad=0.5",fc="white",ec=s_col,alpha=0.9))
    y=0.65
    for r in results:
        ax_sum.text(0.05,y,
                    f"{r['name']}: {r['ai_nc']:.2f}% → {r['ai_tr']:.2f}%  "
                    f"Δ={r['delta']:+.2f}%  ({r['fold']:.2f}×)",
                    transform=ax_sum.transAxes,ha="left",va="top",
                    fontsize=10.5,color="#1B2A4A")
        y-=0.13
    ax_sum.text(0.05,y-0.04,
                f"Paired t-test (n={n}):\n"
                f"t = {t_stat:.3f}   p = {p_val:.4f}   {sig}\n\n"
                f"Area > {AREA_MULTIPLIER}× median single GUV\n"
                f"AND Width > NC P{WID_PCT}\n"
                f"SD of differences = {np.std(diffs,ddof=1):.2f}%",
                transform=ax_sum.transAxes,ha="left",va="top",
                fontsize=9.5,color="#445566",
                bbox=dict(boxstyle="round,pad=0.4",fc="#EBF3F5",ec="#AACCDD",alpha=0.9))
    fig.suptitle(
        f"Cross-Dataset Comparison  |  "
        f"Area > {AREA_MULTIPLIER}× median AND Width > NC P{WID_PCT}  |  "
        f"p = {p_val:.4f} ({sig})",
        fontsize=12,fontweight="bold",y=1.01)
    plt.tight_layout(); plt.show()


---
## Section 4 — Summary Table & GraphPad Export


In [ ]:
rows = []
for r in results:
    d_s = 2*np.sqrt(r["med_s"]/np.pi)
    rows.append({
        "Dataset":               r["name"],
        "channel":               r["mask_id"],
        "n_NC":                  len(r["nc"]),
        "n_TR":                  len(r["tr"]),
        "AI_NC_%":               round(r["ai_nc"],  3),
        "AI_treated_%":          round(r["ai_tr"],  3),
        "delta_%":               round(r["delta"],   3),
        "fold_increase":         round(r["fold"],    3),
        "median_single_area_um2":round(r["med_s"],   2),
        "median_single_diam_um": round(d_s,          2),
        "area_threshold_um2":    round(r["area_t"],  2),
        "width_threshold_um":    round(r["wid_t"],   2),
    })
summary = pd.DataFrame(rows)
display(summary)

if len(results) >= 2:
    t, p = ttest_rel([r["ai_tr"] for r in results],
                     [r["ai_nc"] for r in results])
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))
    print(f"\nPaired t-test: t = {t:.4f}   p = {p:.4f}   ({sig})")

# GraphPad format — paste into a paired column table
cond_col = CONDITION_LABEL.replace(" ","_").replace("µ","u") + "_%"
graphpad = pd.DataFrame({
    "Dataset":            [r["name"] for r in results],
    "Negative_Control_%": [round(r["ai_nc"], 3) for r in results],
    cond_col:             [round(r["ai_tr"], 3) for r in results],
})
print("\nGraphPad format:")
display(graphpad)

# Uncomment to save CSV files:
# summary.to_csv("aggregation_summary.csv", index=False)
# graphpad.to_csv("aggregation_graphpad.csv", index=False)
# print("✓  Saved")
